<a href="https://colab.research.google.com/github/njones61/xslope/blob/main/notebooks/xslope_fem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XSLOPE - Finite Element Method

This notebook illustrates how to use xslope to solve slope stability problems using the finite element method with the Shear Strength Reduction Method (SSRM).

## Install xslope and import functions

In [ ]:
# Install the xslope package with FEM dependencies

%%capture
!apt-get update && apt-get install -y libgl1-mesa-glx libglu1-mesa  # required by gmsh
!pip install xslope[fem]
!pip install gmsh

In [ ]:
# Import functions

from pathlib import Path
import zipfile

from xslope.fileio import load_slope_data
from xslope.mesh import get_material_polygons, build_mesh_from_polygons, export_mesh_to_json, extract_constraint_line_geometry
from xslope.plot import plot_inputs
from xslope.plot_fem import plot_fem_results, plot_fem_data
from xslope.fem import build_fem_data, solve_fem, solve_ssrm, print_reinforcement_summary, print_pile_summary, print_detailed_element_summary, export_fem_solution

## Upload Excel Template

In [ ]:
# Upload Excel input template or zip archive for selected problem.
# Zip archive can include both Excel input template and mesh file from previous analysis.

from google.colab import files
upload = files.upload()
file_name = list(upload.keys())[0]

# See if uploaded file is a zip archive. If so, unzip it
if file_name.endswith('.zip'):
  import zipfile
  with zipfile.ZipFile(file_name, 'r') as zip_ref:
    # Extract all files
    zip_ref.extractall()
    extracted_files = zip_ref.namelist()

    # Find the excel file in the extracted list
    excel_file_found = False
    for f in extracted_files:
      if f.endswith('.xlsx'):
        file_name = f
        excel_file_found = True
        print(f"Found Excel file: {file_name}")
        break

    if not excel_file_found:
        print("Error: No .xlsx file found in the uploaded archive. Please ensure your zip file contains an .xlsx file.")
        file_name = None

## Load slope data

In [ ]:
slope_data = load_slope_data(file_name)
plot_inputs(slope_data, mode='fem', save_png=False)

## Build mesh

In [ ]:
# @title Select meshing options {"run":"auto"}
element_type = "tri6" # @param ["tri3","tri6","quad4","quad8"]
auto_size = True # @param {"type":"boolean"}
size_divisions = 60 # @param {"type":"number"}
target_size = 3 # @param {"type":"number"}
remesh = True # @param {"type":"boolean"}

In [ ]:
input_path = Path(file_name)

# Use existing mesh from slope_data if available, otherwise build a new one
if slope_data.get("mesh") is not None and not remesh:
    print("Using existing mesh file.")
    mesh = slope_data["mesh"]
else:
    print("No existing mesh found in slope_data or remeshing enabled, building new mesh from profile line data.")
    constraint_lines, n_reinf, n_pile = extract_constraint_line_geometry(slope_data)
    polygons = get_material_polygons(slope_data, reinf_lines=constraint_lines)
    print(f"Building mesh with {len(polygons)} polygons, {n_reinf} reinforcement lines, {n_pile} pile lines.")

    if auto_size:
        # find the x-range of the ground_surface and use it to set the target size
        x_range = [min(x for x, _ in slope_data['ground_surface'].coords), max(x for x, _ in slope_data['ground_surface'].coords)]
        target_size = (x_range[1] - x_range[0]) / size_divisions
        print(f"Auto-calculated target element size: {target_size:.3f}")

    mesh = build_mesh_from_polygons(polygons, target_size=target_size, element_type=element_type, lines=constraint_lines)
    mesh_file = input_path.parent / f"{input_path.stem}_mesh.json"
    export_mesh_to_json(mesh, mesh_file)

## Build FEM data and plot mesh

In [ ]:
fem_data = build_fem_data(slope_data, mesh)
plot_fem_data(fem_data, figsize=(14, 7), show_nodes=True, show_bc=True,
              label_elements=False, label_nodes=False, save_png=False)

## Run analysis

In [ ]:
# @title Select analysis options {"run":"auto"}
analysis_type = "ssrm" # @param ["single","ssrm"]
failure_criterion = "non_convergence" # @param ["non_convergence","displacement_limit","displacement_increase","unbalanced_force"]
deform_percent = 15 # @param {"type":"number"}
save_png = True # @param {"type":"boolean"}

F = 1.5     # @param {"type":"number"}
F_min = 1.0 # @param {"type":"number"}
F_max = 2.0 # @param {"type":"number"}

In [ ]:
if analysis_type == "single":
    solution = solve_fem(fem_data, F=F, debug_level=2)
    print(f"  Converged: {solution['converged']}, Iterations: {solution['iterations']}")
    print_reinforcement_summary(fem_data, solution)
    print_pile_summary(fem_data, solution)
    print_detailed_element_summary(fem_data, solution)
    plot_fem_results(fem_data, solution, plot_type=['deformation', 'shear_strain', 'displace_vector'], deform_percent=deform_percent, save_png=save_png)
    export_fem_solution(fem_data, solution, input_path.parent / input_path.stem)

elif analysis_type == "ssrm":
    result = solve_ssrm(fem_data, F_min=F_min, F_max=F_max, tolerance=0.05, debug_level=1,
                        failure_criterion=failure_criterion)
    if result.get("converged", False):
        print(f"\nFactor of Safety: {result['FS']:.2f}")
        print(f"Method: {result.get('method', 'Unknown')}")
        print_reinforcement_summary(fem_data, result['last_solution'])
        print_pile_summary(fem_data, result['last_solution'])
        print_detailed_element_summary(fem_data, result['last_solution'])
        plot_fem_results(fem_data, result['last_solution'],
                         plot_type=['deformation', 'shear_strain', 'displace_vector'], deform_percent=deform_percent, save_png=save_png)
        export_fem_solution(fem_data, result['last_solution'], input_path.parent / input_path.stem)
    else:
        print(f"SSRM failed: {result.get('error', 'Unknown error')}")

## Download results

In [ ]:
# Create a zip file with input, mesh, and FEM result files
mesh_file_name = f"{input_path.stem}_mesh.json"
fem_nodes_file_name = f"{input_path.stem}_fem_nodes.csv"
fem_elements_file_name = f"{input_path.stem}_fem_elements.csv"

zip_file_name = f"{input_path.stem}_fem_results.zip"
with zipfile.ZipFile(zip_file_name, 'w') as zipf:
    zipf.write(file_name)          # Original Excel file
    zipf.write(mesh_file_name)     # Mesh JSON file
    zipf.write(fem_nodes_file_name)
    zipf.write(fem_elements_file_name)

files.download(zip_file_name)